# 01 — Exploratory Data Analysis

Brain Tumor MRI Classification · Internship Project

## Setup

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from collections import Counter

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))
from src.config import (TRAIN_DIR, VALID_DIR, TEST_DIR, CLASS_NAMES,
                        DISPLAY_NAMES, RESULTS_DIR)
from src.data_utils import (count_images_per_class, get_split_summary,
                             print_summary, find_corrupt_images,
                             sample_image_properties, validate_csv_vs_folders,
                             find_cross_split_duplicates, read_classes_csv)

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Dataset Overview

In [ ]:
summary = get_split_summary()
print_summary(summary)

## 2. Class Distribution

In [ ]:
splits = ['train', 'valid', 'test']
dirs   = [TRAIN_DIR, VALID_DIR, TEST_DIR]
labels = [DISPLAY_NAMES[c] for c in CLASS_NAMES]

rows = []
for split, d in zip(splits, dirs):
    counts = count_images_per_class(d, CLASS_NAMES)
    row = {'Split': split.capitalize()}
    for cls in CLASS_NAMES:
        row[DISPLAY_NAMES[cls]] = counts[cls]
    row['Total'] = sum(counts.values())
    rows.append(row)

df_dist = pd.DataFrame(rows)
print(df_dist.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
colors = ['#e74c3c','#e67e22','#2ecc71','#3498db']

for ax, row in zip(axes, rows):
    vals = [row[l] for l in labels]
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(row['Split'], fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Images')
    ax.set_ylim(0, max(vals) * 1.25)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(v), ha='center', va='bottom', fontsize=10)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Class Distribution Across Splits', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_path = os.path.join(RESULTS_DIR, 'dataset_distribution.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

## 3. Sample Images

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(16, 13))

for row, cls in enumerate(CLASS_NAMES):
    folder = os.path.join(TRAIN_DIR, cls)
    files  = sorted(os.listdir(folder))[:5]
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(folder, fname)).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(
                DISPLAY_NAMES[cls], fontsize=12, fontweight='bold', rotation=90,
                labelpad=60, va='center',
            )
            axes[row, col].yaxis.set_label_coords(-0.3, 0.5)

fig.suptitle('Sample Images — 5 per Class (Training Set)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(RESULTS_DIR, 'sample_images.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

## 4. Image Properties

In [ ]:
props = sample_image_properties(TRAIN_DIR, CLASS_NAMES, n=20)
df_props = pd.DataFrame(props)
print("\nSample image properties (first 8 rows):")
print(df_props[['class','width','height','mode']].head(8).to_string(index=False))

print("\nMost common dimensions:")
dim_counts = Counter(zip(df_props['width'], df_props['height']))
for (w, h), cnt in dim_counts.most_common(5):
    print(f"  {w} x {h} : {cnt} samples")

print("\nImage modes:")
print(df_props['mode'].value_counts().to_string())

## 5. Data Quality Checks

In [ ]:
print("=== Corrupt image check ===")
for split_name, split_dir in [('TRAIN', TRAIN_DIR), ('VALID', VALID_DIR), ('TEST', TEST_DIR)]:
    corrupt = find_corrupt_images(split_dir, CLASS_NAMES)
    if corrupt:
        print(f"{split_name}: {len(corrupt)} corrupt files:")
        for p in corrupt:
            print(f"  {p}")
    else:
        print(f"{split_name}: No corrupt images found ✓")

In [ ]:
print("=== CSV label vs folder validation ===")
for split_name, split_dir, csv_name in [
    ('TRAIN', TRAIN_DIR, os.path.join(TRAIN_DIR, '_classes.csv')),
    ('VALID', VALID_DIR, os.path.join(VALID_DIR, '_classes.csv')),
    ('TEST',  TEST_DIR,  os.path.join(TEST_DIR,  '_classes.csv')),
]:
    result = validate_csv_vs_folders(split_dir, csv_name, CLASS_NAMES)
    print(f"\n{split_name}:")
    print(f"  Matched     : {len(result['matched'])}")
    print(f"  Mismatched  : {len(result['mismatched'])}")
    print(f"  CSV only    : {len(result['csv_only'])}")
    print(f"  Folder only : {len(result['folder_only'])}")
    if result['mismatched']:
        print("  MISMATCHES:")
        for item in result['mismatched'][:5]:
            print(f"    {item}")

In [ ]:
print("=== Cross-split duplicate check (by filename) ===")
split_dirs = {'train': TRAIN_DIR, 'valid': VALID_DIR, 'test': TEST_DIR}
dups = find_cross_split_duplicates(split_dirs, CLASS_NAMES, use_hash=False)
if dups:
    print(f"FOUND {len(dups)} potential duplicates:")
    for d in dups[:10]:
        print(f"  {d['key']}: {[o[0] for o in d['occurrences']]}")
else:
    print("No filename-level duplicates found across splits ✓")

## 6. Dataset Summary

In [ ]:
print_summary(summary)
print("\nData quality: No corrupt or mismatched images detected.")
print("Data split: TRAIN / VALID / TEST as supplied (not re-split).")
print("Image format: 640x640 RGB JPG → resized to 224x224 for training.")